<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/PytochCNNPractice2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms,datasets
from torch.utils.data import DataLoader

import os
from google.colab import userdata

In [6]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('username and key activated')

username and key activated


In [7]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:48<00:00, 83.2MB/s]



In [8]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

In [4]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [9]:
#transform
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip,
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip,
    transforms.ToTensor()
])

In [11]:
#dataset and dataload
train_data = datasets.ImageFolder(root=train_dir,transform=train_transform)
val_data = datasets.ImageFolder(root=validation_dir,transform=val_transform)

train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
val_loader = DataLoader(dataset=val_data,batch_size=32,num_workers=2,pin_memory=True)

In [12]:
#model
class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv = nn.Sequential(
        nn.Conv2d(3,32,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(32,32,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(64,64,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.flatten = nn.Flatten()

    self.fc = nn.Sequential(
        nn.LazyLinear(128),
        nn.ReLU(),
        nn.Linear(128,1)
    )

  def forward(self,x):
    x = self.conv(x)
    x = self.flatten(x)
    x = self.fc(x)

    return x

In [15]:
model = CNN().to(device)

In [16]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [17]:
#training

epochs = 5

for epoch in range(epochs):
  model.train()

  train_loss = 0
  train_correct = 0
  train_total = 0

  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.float().unsqueeze(1).to(device)

    optimizer.zero_grad()

    output = model(images)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()

    preds = (torch.sigmoid(output) > 0.5).int()
    train_correct += (preds.squeeze() == labels.squeeze()).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_correct / train_total

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images,labels in val_loader:
      images = images.to(device)
      labels = labels.float().unsqueeze(1).to(device)

      optimizer.zero_grad()

      output = model(images)
      loss = loss_fn(output,labels)

      val_loss += loss.item()

      preds = (torch.sigmoid(output) > 0.5).int()
      val_correct += (preds.squeeze() == labels.squeeze()).sum().item()
      val_total += labels.size(0)

  val_accuracy = train_correct / train_total

  print(f"Epochs {epoch+1}/{epochs}")



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 247, in __getitem__
    sample = self.transform(sample)
             ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/transforms/transforms.py", line 95, in __call__
    img = t(img)
          ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/transforms/transforms.py", line 137, in __call__
    return F.to_tensor(pic)
           ^^^^^^^^^^^^^^^^
  Fil